# HyDE-Enhanced Agentic Retrieval — Kaggle Submission

Production notebook for the Omnilex Legal Retrieval competition.  
Pipeline: FAISS semantic + HyDE + ReAct Agent

## Requirements
- **Kaggle Accelerator**: GPU T4 x2
- **Internet**: ON (for pip installs)
- **Datasets attached**:
  - Competition data (has train.csv, test.csv, laws_de.csv, court_considerations.csv)
  - Mistral 7B GGUF model dataset

In [1]:
# === CELL 1: INSTALL DEPENDENCIES ===
!pip install -q sentence-transformers faiss-cpu pydantic transformers rank-bm25
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 979.5 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00


In [2]:
print("hello")

hello


In [3]:
import os

# 1. All input datasets (competition data + attached datasets)
print("=" * 60)
print("/kaggle/input/ (all attached datasets)")
print("=" * 60)
for root, dirs, files in os.walk("/kaggle/input/"):
    for f in files:
        full = os.path.join(root, f)
        print(f"  {full} — {os.path.getsize(full)/1e6:.1f} MB")

# 2. Working directory (your outputs + cache)
print("\n" + "=" * 60)
print("/kaggle/working/ (outputs + cache)")
print("=" * 60)
for root, dirs, files in os.walk("/kaggle/working/"):
    for f in files:
        full = os.path.join(root, f)
        print(f"  {full} — {os.path.getsize(full)/1e6:.1f} MB")

# 3. Temp files (sometimes leftover from previous cells)
print("\n" + "=" * 60)
print("/tmp/ (top-level only)")
print("=" * 60)
for f in os.listdir("/tmp/"):
    full = f"/tmp/{f}"
    if os.path.isfile(full):
        print(f"  {full} — {os.path.getsize(full)/1e6:.1f} MB")
    else:
        print(f"  {full}/ (dir)")

/kaggle/input/ (all attached datasets)
  /kaggle/input/datasets/charan1996/rag-checkpoints/corpus_documents.pkl — 242.6 MB
  /kaggle/input/datasets/charan1996/mistral-7b-gguf/mistral-7b-instruct-v0.2.Q4_K_M.gguf — 4368.4 MB
  /kaggle/input/competitions/llm-agentic-legal-information-retrieval/sample_submission.csv — 0.0 MB
  /kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv — 73.0 MB
  /kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv — 0.0 MB
  /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv — 2425.8 MB
  /kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv — 2.0 MB
  /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv — 0.1 MB

/kaggle/working/ (outputs + cache)
  /kaggle/working/.virtual_documents/__notebook_source__.ipynb — 0.0 MB

/tmp/ (top-level only)
  /tmp/uv-setuptools-a84f10b927b58cc7.lock — 0.0 MB
  /tmp/clean-layer.sh — 0.0 MB
  /

In [4]:
# CHECK: What cached files exist on Kaggle from previous runs?
import os
from datetime import datetime

def show_dir(path, label):
    print(f"\n=== {label}: {path} ===")
    if not os.path.exists(path):
        print("  ❌ Directory does not exist")
        return
    for f in sorted(os.listdir(path)):
        full = os.path.join(path, f)
        if os.path.isfile(full):
            size = os.path.getsize(full)
            mtime = datetime.fromtimestamp(os.path.getmtime(full)).strftime("%Y-%m-%d %H:%M")
            print(f"  📄 {f} — {size/1e6:.1f} MB — modified: {mtime}")
        elif os.path.isdir(full):
            sub_files = os.listdir(full)
            print(f"  📁 {f}/ ({len(sub_files)} items)")
            for sf in sorted(sub_files)[:10]:
                sfull = os.path.join(full, sf)
                if os.path.isfile(sfull):
                    sz = os.path.getsize(sfull)
                    mt = datetime.fromtimestamp(os.path.getmtime(sfull)).strftime("%Y-%m-%d %H:%M")
                    print(f"      {sf} — {sz/1e6:.1f} MB — modified: {mt}")

show_dir("/kaggle/working", "Working dir")
show_dir("/kaggle/working/cache", "Cache dir (INDEX_PATH)")

# Also check if there's a persistent dataset with pre-built indices
print("\n=== All /kaggle/input/ datasets ===")
for item in sorted(os.listdir("/kaggle/input/")):
    subpath = f"/kaggle/input/{item}"
    if os.path.isdir(subpath):
        for sub in sorted(os.listdir(subpath)):
            subpath2 = os.path.join(subpath, sub)
            if os.path.isdir(subpath2):
                files = os.listdir(subpath2)
                pkl_files = [f for f in files if f.endswith('.pkl') or f.endswith('.npy') or f.endswith('.faiss')]
                if pkl_files:
                    print(f"  📁 {item}/{sub}/ — has cached files: {pkl_files}")


=== Working dir: /kaggle/working ===
  📁 .virtual_documents/ (1 items)
      __notebook_source__.ipynb — 0.0 MB — modified: 2026-05-26 02:20

=== Cache dir (INDEX_PATH): /kaggle/working/cache ===
  ❌ Directory does not exist

=== All /kaggle/input/ datasets ===


In [5]:
# === CELL 2: IMPORTS, PATHS & CONFIGURATION ===
import os, sys, re, gc, time, pickle, hashlib, shutil, subprocess, json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.notebook import tqdm

# ---- Paths (Kaggle only) ----
DATA_PATH = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")
MODEL_PATH = Path("/kaggle/input/datasets/charan1996/mistral-7b-gguf")
CHECKPOINT_DS = Path("/kaggle/input/datasets/charan1996/rag-checkpoints")  # persistent read
OUTPUT_PATH = Path("/kaggle/working")
INDEX_PATH = Path("/kaggle/working/cache")

# ---- Corpus CSV paths ----
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"
TRAIN_CSV = DATA_PATH / "train.csv"
TEST_CSV = DATA_PATH / "test.csv"

# ---- Index cache paths ----
FAISS_LAWS_PATH = INDEX_PATH / "faiss_laws_qwen3_embeddings.pkl"
FAISS_COURTS_PATH = INDEX_PATH / "faiss_courts_qwen3_embeddings.pkl"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

# ---- Configuration ----
CONFIG = {
    "model_file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    "n_ctx": 8192,
    "n_threads": 8,
    "n_gpu_layers": -1,
    "max_iterations": 5,
    "max_tokens": 512,
    "temperature": 0.1,
    "max_observation_chars": 1200,
    "max_conversation_chars": 28000,
    "top_k_laws": 50,
    "top_k_courts": 50,
    "hyde_max_tokens": 300,
    "hyde_temperature": 0.3,
    "hyde_target_chars_law": 500,
    "hyde_target_chars_court": 600,
    "hyde_enabled": True,
    "prf_top_k": 5,  # Pseudo-Relevance Feedback: initial results for HyDE context
    "rerank_model": "Qwen/Qwen3-Reranker-0.6B",
    "rerank_top_n": 50,
    "final_rerank_top_n": 50,
    "rerank_enabled": True,
    "rerank_per_search": False,  # DISABLED: per-search reranker was destroying good RRF results
    # ---- Embedding model (Qwen3-Embedding-0.6B) ----
    "embed_model": "Qwen/Qwen3-Embedding-0.6B",
    "embed_dim": 1024,
    "embed_max_length": 1024,  # tokens (Qwen3 supports 8192; 1024 covers ~2800 chars of German)
    # ---- Instruction prompts for embedding phases ----
    "prompt_doc_law": "Instruct: Represent this Swiss federal statute article in German for legal citation retrieval\nDocument: ",
    "prompt_doc_court": "Instruct: Represent this Swiss Federal Court decision excerpt in German for legal citation retrieval\nDocument: ",
    "prompt_query_law": "Instruct: Given German legal search terms, retrieve relevant Swiss federal statute articles from the SR collection\nQuery: ",
    "prompt_query_court": "Instruct: Given German legal search terms, retrieve relevant Swiss Federal Court decisions (BGE)\nQuery: ",
    "prompt_hyde_law": "Instruct: Given a hypothetical Swiss legal text in German, retrieve real Swiss federal statutes with similar legal content\nQuery: ",
    "prompt_hyde_court": "Instruct: Given a hypothetical Swiss court decision text in German, retrieve real Swiss Federal Court decisions with similar content\nQuery: ",
}

# ---- Validation ----
assert LAWS_CSV.exists(), f"Laws CSV not found: {LAWS_CSV}"
assert COURTS_CSV.exists(), f"Courts CSV not found: {COURTS_CSV}"
assert TEST_CSV.exists(), f"Test CSV not found: {TEST_CSV}"

print(f"Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)")
print(f"Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)")
print(f"Test CSV: {TEST_CSV}")

# ---- Restore from persistent dataset (if available from previous run) ----
if CHECKPOINT_DS.exists():
    restored = 0
    for f in CHECKPOINT_DS.iterdir():
        if f.suffix == '.pkl':
            dest = INDEX_PATH / f.name
            if not dest.exists():
                shutil.copy2(f, dest)
                restored += 1
    if restored:
        print(f"♻️  Restored {restored} cached files from {CHECKPOINT_DS}")
    else:
        print(f"✅ Checkpoint dataset found — caches already in place")
else:
    print(f"No checkpoint dataset yet — will build from scratch")

# ---- Save helpers ----
# /kaggle/working/ = TEMPORARY (wiped when session expires)
# kaggle.com/datasets/charan1996/rag-checkpoints = PERMANENT (survives forever)
# Every _save_checkpoint() pushes to the permanent dataset.
# If the push FAILS → notebook CRASHES immediately. No silent data loss.
DATASET_SLUG = "charan1996/rag-checkpoints"
_DATASET_STAGING = OUTPUT_PATH / "_dataset_staging"
_push_count = 0

def _push_to_dataset():
    """Push all checkpoint files to Kaggle Dataset.
    RAISES RuntimeError on failure — notebook STOPS so you know data is NOT saved."""
    global _push_count
    all_paths = [
        FAISS_LAWS_PATH, FAISS_COURTS_PATH,
        INDEX_PATH / "corpus_documents.pkl", INDEX_PATH / "hyde_cache.pkl",
        OUTPUT_PATH / "predictions_checkpoint.pkl",
        OUTPUT_PATH / "val_predictions_checkpoint.pkl",
        OUTPUT_PATH / "submission.csv",
    ]
    existing_files = [Path(p) for p in all_paths if Path(p).exists()]
    if not existing_files:
        print("⚠️  No files to push yet")
        return
    # Stage files
    if _DATASET_STAGING.exists():
        shutil.rmtree(_DATASET_STAGING)
    _DATASET_STAGING.mkdir(parents=True, exist_ok=True)
    for fpath in existing_files:
        shutil.copy2(fpath, _DATASET_STAGING / fpath.name)
    metadata = {"id": DATASET_SLUG, "title": "rag-checkpoints", "licenses": [{"name": "CC0-1.0"}]}
    with open(_DATASET_STAGING / "dataset-metadata.json", "w") as f:
        json.dump(metadata, f)
    _push_count += 1
    # Push to Kaggle
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(_DATASET_STAGING),
         "-m", f"checkpoint #{_push_count}", "--dir-mode", "zip"],
        capture_output=True, text=True, timeout=300
    )
    # If dataset doesn't exist yet, create it
    if result.returncode != 0 and ("404" in result.stderr or "not found" in result.stderr.lower() or "403" in result.stdout or "Forbidden" in result.stdout):
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(_DATASET_STAGING)],
            capture_output=True, text=True, timeout=300
        )
    # HARD CHECK — did it work?
    if result.returncode != 0:
        msg = f"❌ DATASET PUSH FAILED!\n  stdout: {result.stdout}\n  stderr: {result.stderr}"
        print(msg)
        raise RuntimeError(msg)
    file_list = ", ".join(f.name for f in existing_files)
    print(f"✅ SAVED to kaggle.com/datasets/{DATASET_SLUG} (push #{_push_count}): {file_list}")


def _save_checkpoint(*files):
    """Copy files to /kaggle/working/ then push ALL checkpoints to permanent dataset.
    If push fails → notebook CRASHES (so you know it's not saved)."""
    for fpath in files:
        fpath = Path(fpath)
        if not fpath.exists():
            continue
        dest = OUTPUT_PATH / fpath.name
        if dest != fpath:
            shutil.copy2(fpath, dest)
    _push_to_dataset()


def _final_save():
    """Final push + print summary."""
    _push_to_dataset()
    print(f"\n{'='*60}")
    print(f"  ✅ ALL FILES PERMANENTLY SAVED")
    print(f"  Location: kaggle.com/datasets/{DATASET_SLUG}")
    print(f"  Files:")
    for f in sorted(OUTPUT_PATH.iterdir()):
        if f.is_file() and not f.name.startswith("_"):
            print(f"     {f.name} ({f.stat().st_size/1e6:.1f} MB)")
    print(f"{'='*60}")

print(f"💾 Permanent save target: kaggle.com/datasets/{DATASET_SLUG}")
print(f"   If push fails → notebook STOPS (no silent data loss)")

Laws CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv (73.0 MB)
Courts CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv (2.43 GB)
Test CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv
♻️  Restored 1 cached files from /kaggle/input/datasets/charan1996/rag-checkpoints
💾 Permanent save target: kaggle.com/datasets/charan1996/rag-checkpoints
   If push fails → notebook STOPS (no silent data loss)


In [6]:
# === CELL 3: CORPUS LOADING ===

CORPUS_CACHE_PATH = INDEX_PATH / "corpus_documents.pkl"


def load_csv_corpus(csv_path, chunk_size=100_000, max_rows=None):
    """Load CSV corpus into list of dicts."""
    documents = []
    total_rows = sum(len(c) for c in pd.read_csv(csv_path, usecols=["citation"], chunksize=100_000))
    if max_rows:
        total_rows = min(total_rows, max_rows)
    rows_loaded = 0
    with tqdm(total=total_rows, desc=f"Loading {csv_path.name}") as pbar:
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
            for _, row in chunk.iterrows():
                if max_rows and rows_loaded >= max_rows:
                    break
                documents.append({
                    "citation": str(row["citation"]),
                    "text": str(row["text"]) if pd.notna(row["text"]) else ""
                })
                rows_loaded += 1
            pbar.update(min(len(chunk), total_rows - pbar.n))
            if max_rows and rows_loaded >= max_rows:
                break
    return documents


def load_csv_corpus_sampled(csv_path, sample_n):
    """Load CSV corpus with random sampling (avoids positional bias)."""
    print(f"  Random sampling {sample_n:,} rows from {csv_path.name}...")
    df = pd.read_csv(csv_path, usecols=["citation", "text"], dtype={"citation": str, "text": str}, na_filter=False)
    total = len(df)
    if total <= sample_n:
        print(f"  Corpus has only {total:,} rows — using all")
    else:
        df = df.sample(n=sample_n, random_state=42)
        print(f"  Sampled {sample_n:,} from {total:,} total")
    documents = [{"citation": r["citation"], "text": r["text"]} for _, r in df.iterrows()]
    return documents


def get_or_load_corpus(name, csv_path, cache_path, max_rows=None, sample_n=None):
    """Load cached corpus or build from CSV."""
    if cache_path.exists():
        print(f"Loading cached {name} corpus from {cache_path}")
        with open(cache_path, 'rb') as f:
            data = pickle.load(f)
        documents = data[name]
        print(f"  Loaded {len(documents):,} documents")
        return documents
    if not csv_path.exists():
        return []
    print(f"Loading {name} corpus from {csv_path}")
    if sample_n:
        documents = load_csv_corpus_sampled(csv_path, sample_n=sample_n)
    else:
        documents = load_csv_corpus(csv_path, max_rows=max_rows)
    print(f"  Loaded {len(documents):,} documents")
    return documents


# ---- Load Corpora ----
if CORPUS_CACHE_PATH.exists():
    print(f"Loading cached corpora from {CORPUS_CACHE_PATH}")
    with open(CORPUS_CACHE_PATH, 'rb') as f:
        _corpus_data = pickle.load(f)
    laws_documents = _corpus_data["laws"]
    courts_documents = _corpus_data["courts"]
    print(f"  Laws: {len(laws_documents):,}, Courts: {len(courts_documents):,}")
    del _corpus_data
else:
    laws_documents = load_csv_corpus(LAWS_CSV)
    print(f"Laws corpus: {len(laws_documents):,} documents")

    courts_documents = load_csv_corpus_sampled(COURTS_CSV, sample_n=200_000)
    print(f"Courts corpus: {len(courts_documents):,} documents")

    # Cache for fast reload
    with open(CORPUS_CACHE_PATH, 'wb') as f:
        pickle.dump({"laws": laws_documents, "courts": courts_documents}, f)
    print(f"  Cached to {CORPUS_CACHE_PATH}")

_save_checkpoint(CORPUS_CACHE_PATH)

Loading cached corpora from /kaggle/working/cache/corpus_documents.pkl
  Laws: 175,933, Courts: 200,000
✅ SAVED to kaggle.com/datasets/charan1996/rag-checkpoints (push #1): corpus_documents.pkl


In [7]:
# === CELL 4: LOAD LLM (pinned to GPU 0) ===
import torch
torch.cuda.set_device(0)  # LLM on GPU 0

from llama_cpp import Llama

# Find model file
model_file = MODEL_PATH / CONFIG["model_file"]
if not model_file.exists():
    gguf_files = list(MODEL_PATH.rglob("*.gguf"))
    if gguf_files:
        model_file = gguf_files[0]
    else:
        raise FileNotFoundError(f"No GGUF model found in {MODEL_PATH}")

print(f"Loading model: {model_file.name} (GPU 0)")
llm = Llama(
    model_path=str(model_file),
    n_ctx=CONFIG["n_ctx"],
    n_threads=CONFIG["n_threads"],
    n_gpu_layers=CONFIG["n_gpu_layers"],
    main_gpu=0,
    verbose=False,
)
gc.collect()
print(f"Model loaded on GPU 0 — GPU layers: {CONFIG['n_gpu_layers']}")


Loading model: mistral-7b-instruct-v0.2.Q4_K_M.gguf (GPU 0)


llama_context: n_ctx_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded on GPU 0 — GPU layers: -1


In [8]:
# === CELL 5: TYPE HELPERS (for corpus type labeling) ===
# The few-shot bank is REMOVED — replaced by PRF (Pseudo-Relevance Feedback).
# We keep type extraction helpers because they're used for laws_doc_types/courts_doc_types.

def get_law_type(citation):
    match = re.search(r'\b([A-Z]{2,}[a-z]?)\s*$', citation.strip())
    if match:
        return match.group(1)
    matches = re.findall(r'\b([A-Z]{2,})\b', citation)
    return matches[-1] if matches else "OTHER"

def get_court_type(citation):
    m = re.match(r'BGE\s+\d+\s+([IVX]+)', citation)
    if m:
        return f"BGE_{m.group(1)}"
    m = re.match(r'(\d+[A-Z]+)', citation)
    if m:
        return f"CASE_{m.group(1)}"
    return "OTHER"

print("Type helpers ready (few-shot bank REMOVED — using PRF approach)")
print("  PRF: initial raw search → top-k context → HyDE generation → final search")


Type helpers ready (few-shot bank REMOVED — using PRF approach)
  PRF: initial raw search → top-k context → HyDE generation → final search


In [9]:
# === CELL 6: TYPE REGISTRY ===

# Compute doc_types arrays
laws_doc_types = np.array([get_law_type(d["citation"]) for d in laws_documents], dtype=object)
courts_doc_types = np.array([get_court_type(d["citation"]) for d in courts_documents], dtype=object)

law_type_counts = Counter(laws_doc_types)
court_type_counts = Counter(courts_doc_types)

# Build type strings for agent prompt
LAW_TYPES_FOR_PROMPT = ", ".join(
    f"{t}({c})" for t, c in sorted(law_type_counts.items(), key=lambda x: -x[1])[:40]
)
COURT_TYPES_FOR_PROMPT = ", ".join(
    f"{t}({c})" for t, c in sorted(court_type_counts.items(), key=lambda x: -x[1])
)

print(f"Type registry: {len(law_type_counts)} law types, {len(court_type_counts)} court types")

Type registry: 656 law types, 58 court types


In [11]:
# === CELL 7: FAISS SEMANTIC SEARCH + QWEN3-EMBEDDING ===
import faiss
import torch
from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {CONFIG['embed_model']}...")
_st_model = SentenceTransformer(
    CONFIG["embed_model"],
    device="cuda:1",
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.float16},  # fp16 = half VRAM
)
_st_model.max_seq_length = CONFIG["embed_max_length"]
print(f"  Model loaded (fp16): dim={_st_model.get_sentence_embedding_dimension()}, device=cuda:1")
print(f"  Max seq length: {_st_model.max_seq_length}")

# ---- Embed main corpus for search (with type-specific document prompts) ----
# NOTE: Reranker loaded AFTER embedding to avoid OOM (both can't fit during batch encoding)
def embed_corpus(documents, doc_type="law", desc="Embedding", batch_size=8):
    """Embed corpus with type-specific document instruction prompt."""
    texts = [f"{d.get('citation','')}: {d.get('text','')[:1500]}" for d in documents]
    prompt = CONFIG["prompt_doc_law"] if doc_type == "law" else CONFIG["prompt_doc_court"]
    print(f"  {desc}: {len(texts):,} docs, batch_size={batch_size}")
    print(f"  Prompt: \"{prompt[:60]}...\"")
    t0 = time.time()
    embeddings = _st_model.encode(
        texts,
        prompt=prompt,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=batch_size,
    )
    print(f"  Done in {time.time()-t0:.1f}s ({len(texts)/(time.time()-t0):.0f} docs/sec)")
    return embeddings.astype('float32')

# Laws embeddings
if FAISS_LAWS_PATH.exists():
    print(f"Loading cached law embeddings (Qwen3)...")
    with open(FAISS_LAWS_PATH, 'rb') as f:
        law_corpus_embeddings = pickle.load(f)
else:
    law_corpus_embeddings = embed_corpus(laws_documents, doc_type="law", desc="Laws")
    with open(FAISS_LAWS_PATH, 'wb') as f:
        pickle.dump(law_corpus_embeddings, f)
    _save_checkpoint(FAISS_LAWS_PATH)

# Courts embeddings
if FAISS_COURTS_PATH.exists():
    print(f"Loading cached court embeddings (Qwen3)...")
    with open(FAISS_COURTS_PATH, 'rb') as f:
        court_corpus_embeddings = pickle.load(f)
else:
    court_corpus_embeddings = embed_corpus(courts_documents, doc_type="court", desc="Courts")
    with open(FAISS_COURTS_PATH, 'wb') as f:
        pickle.dump(court_corpus_embeddings, f)
    _save_checkpoint(FAISS_COURTS_PATH)

# Build corpus FAISS indices
dim = _st_model.get_sentence_embedding_dimension()
faiss_law_index = faiss.IndexFlatIP(dim)
faiss_law_index.add(law_corpus_embeddings)

faiss_court_index = faiss.IndexFlatIP(dim)
faiss_court_index.add(court_corpus_embeddings)

# Free raw embeddings from memory
del law_corpus_embeddings, court_corpus_embeddings
gc.collect()
torch.cuda.empty_cache()

# ---- NOW load Qwen3-Reranker (generative yes/no, NOT CrossEncoder) ----
# WHY: CrossEncoder loads Qwen3 as sequence classification with random head = garbage scores.
# Qwen3-Reranker is a causal LM that judges relevance by generating "yes" or "no".
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F

RERANKER_TASK_INSTRUCTION = (
    "Given a legal question, retrieve all relevant Swiss legal citations "
    "including federal statutes (SR) and Federal Court decisions (BGE)"
)
TEXT_TRUNCATE = 2048

class Qwen3Reranker:
    """Qwen3-Reranker-0.6B using correct AutoModelForCausalLM approach."""

    SYSTEM_PROMPT = (
        'Judge whether the Document meets the requirements based on the '
        'Query and the Instruct provided. Note that the answer can only be '
        '"yes" or "no".'
    )

    def __init__(self, model_name, device='cuda', dtype=torch.float16):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name, padding_side='left', trust_remote_code=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype, trust_remote_code=True,
        ).to(device).eval()
        self.device = device
        self.yes_id = self.tokenizer.convert_tokens_to_ids('yes')
        self.no_id = self.tokenizer.convert_tokens_to_ids('no')

    def _format_pair(self, instruction, query, document):
        return (
            f'<|im_start|>system\n{self.SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n'
            f'<Instruct>: {instruction}\n'
            f'<Query>: {query}\n'
            f'<Document>: {document}\n'
            f'<|im_end|>\n'
            f'<|im_start|>assistant\n<think>\n\n</think>\n\n'
        )

    @torch.no_grad()
    def predict(self, pairs, batch_size=8, instruction=None, show_progress_bar=False):
        """Score query-document pairs. Returns list of relevance scores (0-1)."""
        if instruction is None:
            instruction = RERANKER_TASK_INSTRUCTION
        all_scores = []
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]
            prompts = [
                self._format_pair(instruction, q, d[:TEXT_TRUNCATE])
                for q, d in batch
            ]
            inputs = self.tokenizer(
                prompts, padding=True, truncation=True,
                max_length=4096, return_tensors='pt',
            ).to(self.device)
            logits = self.model(**inputs).logits[:, -1, :]
            yes_no = torch.stack(
                [logits[:, self.no_id], logits[:, self.yes_id]], dim=1
            )
            probs = F.softmax(yes_no, dim=1)
            scores = probs[:, 1].cpu().numpy()  # P(yes)
            all_scores.extend(scores.tolist())
        return all_scores


# ---- Load reranker ----
_reranker = None
if CONFIG["rerank_enabled"]:
    try:
        print(f"Loading reranker: {CONFIG['rerank_model']} (Qwen3 generative yes/no)...")
        _reranker = Qwen3Reranker(CONFIG["rerank_model"], device="cuda:1")
        print(f"  Reranker loaded on cuda:1 (P(yes) scoring)")
    except Exception as e:
        print(f"  \u26a0\ufe0f Reranker failed to load: {e}")
        _reranker = None
        print(f"  Continuing WITHOUT reranker (FAISS-only retrieval)")
else:
    print("  Reranker: DISABLED")

print(f"  Embedding model: {CONFIG['embed_model']} ({dim}d, fp16)")
print(f"  Document prompts: law-specific + court-specific")
print(f"\nCorpus FAISS: laws={faiss_law_index.ntotal:,}, courts={faiss_court_index.ntotal:,} vectors ({dim}d)")
_save_checkpoint(FAISS_LAWS_PATH, FAISS_COURTS_PATH)


Loading embedding model: Qwen/Qwen3-Embedding-0.6B...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Model loaded (fp16): dim=1024, device=cuda:1
  Max seq length: 1024
  Laws: 175,933 docs, batch_size=8
  Prompt: "Instruct: Represent this Swiss federal statute article in Ge..."


Batches:   0%|          | 0/21992 [00:00<?, ?it/s]

: 

In [ ]:
# === CELL 7b: BM25 KEYWORD INDEX ===
# BM25 (Okapi BM25) for exact keyword matching — complements FAISS semantic search.
# Combined via Reciprocal Rank Fusion (RRF) for hybrid retrieval.

from rank_bm25 import BM25Okapi
import re as _re

def _tokenize_german(text):
    """Simple German-aware tokenizer: lowercase, split on non-alphanumeric, keep umlauts."""
    text = text.lower()
    tokens = _re.findall(r'[a-zäöüß\d]+', text)
    # Remove very short tokens (articles, prepositions) that add noise
    return [t for t in tokens if len(t) > 2]


# Build BM25 index for laws
print("Building BM25 index for laws...")
_law_bm25_texts = [
    f"{d.get('citation', '')} {d.get('text', '')}" for d in laws_documents
]
_law_bm25_corpus = [_tokenize_german(t) for t in _law_bm25_texts]
bm25_law_index = BM25Okapi(_law_bm25_corpus)
del _law_bm25_texts, _law_bm25_corpus
print(f"  Laws BM25: {len(laws_documents):,} documents indexed")

# Build BM25 index for courts
print("Building BM25 index for courts...")
_court_bm25_texts = [
    f"{d.get('citation', '')} {d.get('text', '')}" for d in courts_documents
]
_court_bm25_corpus = [_tokenize_german(t) for t in _court_bm25_texts]
bm25_court_index = BM25Okapi(_court_bm25_corpus)
del _court_bm25_texts, _court_bm25_corpus
print(f"  Courts BM25: {len(courts_documents):,} documents indexed")

gc.collect()
print(f"\nBM25 indices ready (used for hybrid retrieval with FAISS via RRF)")


In [ ]:
# === CELL 8: PRF-BASED HyDE GENERATION ===
# Pseudo-Relevance Feedback: use top-K raw search results as context for HyDE.
# No external few-shot bank needed — the corpus IS the context source.

# How many queries to show detailed output for (then goes quiet)
_VERBOSE_FIRST_N = 5
_query_count = 0

# ---- HyDE Cache (disk-backed) ----
HYDE_CACHE_PATH = INDEX_PATH / "hyde_cache.pkl"

# Start fresh cache — old entries used different keys (type_hints-based)
_hyde_cache = {}
_hyde_cache_dirty = 0
print(f"HyDE cache: starting fresh (PRF approach, old cache invalidated)")

def _save_hyde_cache():
    """Persist HyDE cache to disk."""
    global _hyde_cache_dirty
    with open(HYDE_CACHE_PATH, 'wb') as f:
        pickle.dump(_hyde_cache, f)
    _hyde_cache_dirty = 0


def build_hyde_prompt(query, doc_type="law", prf_snippets=None):
    """Build prompt for hypothetical German legal document generation.
    
    prf_snippets: list of text strings from top-K initial search results (actual corpus docs).
    These ground the HyDE generation in real corpus vocabulary and structure.
    """
    if doc_type == "law":
        instruction = (
            "Du bist ein Schweizer Rechtsexperte. Gegeben eine rechtliche Frage, "
            "schreibe einen hypothetischen Schweizer Gesetzesartikel auf Deutsch, "
            "der diese Frage beantworten würde.\n"
            f"Der Text soll ca. {CONFIG['hyde_target_chars_law']} Zeichen lang sein.\n"
            "Antworte ausschliesslich auf Deutsch, auch wenn die Frage auf Englisch ist.\n"
            "Keine Erklärungen, kein Vorwort — nur den Gesetzestext."
        )
    else:
        instruction = (
            "Du bist ein Schweizer Rechtsexperte. Gegeben eine rechtliche Frage, "
            "schreibe eine hypothetische Erwägung eines Schweizer Bundesgerichtsentscheids "
            "auf Deutsch, die diese Frage behandeln würde.\n"
            f"Der Text soll ca. {CONFIG['hyde_target_chars_court']} Zeichen lang sein.\n"
            "Antworte ausschliesslich auf Deutsch, auch wenn die Frage auf Englisch ist.\n"
            "Keine Erklärungen, kein Vorwort — nur den Erwägungstext."
        )

    # PRF context: actual corpus documents as style/vocabulary reference
    context_text = ""
    if prf_snippets:
        context_text = "\n\nRelevante Referenztexte aus dem Korpus:\n"
        for i_ref, snippet in enumerate(prf_snippets[:CONFIG["prf_top_k"]]):
            context_text += f"\nReferenz {i_ref+1}: {snippet}\n"
        context_text += "\nSchreibe einen ähnlichen Text, der die folgende Frage beantwortet:\n"

    return f"[INST] {instruction}{context_text}\n\nFrage: {query}\n\nHypothetischer Text: [/INST]"


def generate_hypothetical_document(query, doc_type="law", prf_snippets=None):
    """Generate hypothetical German legal document for HyDE using PRF context.
    
    prf_snippets: text snippets from initial raw FAISS search (top-K results).
    These provide real corpus vocabulary/structure to ground the generation.
    """
    global _hyde_cache_dirty
    if not CONFIG.get("hyde_enabled", True):
        return query

    # Cache key: query + doc_type + hash of PRF snippets
    prf_hash = hashlib.md5("||".join(prf_snippets or []).encode()).hexdigest()[:8]
    cache_key = hashlib.md5(f"{query}:{doc_type}:prf:{prf_hash}".encode()).hexdigest()
    if cache_key in _hyde_cache:
        hyde_doc = _hyde_cache[cache_key]
        if _query_count < _VERBOSE_FIRST_N:
            print(f"    [HyDE {doc_type}] CACHE HIT → {len(hyde_doc)} chars: \"{hyde_doc[:80]}...\"")
        return hyde_doc

    prompt = build_hyde_prompt(query, doc_type, prf_snippets=prf_snippets)
    try:
        response = llm(
            prompt,
            max_tokens=CONFIG["hyde_max_tokens"],
            temperature=CONFIG["hyde_temperature"],
            stop=["[INST]", "</s>", "\nFrage:", "\n\nFrage:"],
        )
        hyde_doc = response["choices"][0]["text"].strip()
    except Exception:
        hyde_doc = query

    _hyde_cache[cache_key] = hyde_doc
    _hyde_cache_dirty += 1

    if _query_count < _VERBOSE_FIRST_N:
        print(f"    [HyDE {doc_type}] PRF-GENERATED → {len(hyde_doc)} chars: \"{hyde_doc[:80]}...\"")

    # Auto-save every 50 new entries
    if _hyde_cache_dirty >= 50:
        _save_hyde_cache()

    return hyde_doc


print(f"PRF-based HyDE generation ready")
print(f"  prf_top_k={CONFIG['prf_top_k']} (initial results used as context)")
print(f"  hyde_max_tokens={CONFIG['hyde_max_tokens']}, temperature={CONFIG['hyde_temperature']}")


In [ ]:
# === CELL 9: HYBRID SEARCH (FAISS + BM25 + RRF) ===
# Reciprocal Rank Fusion combines FAISS semantic + BM25 keyword rankings.
# RRF_score(d) = sum(1 / (k + rank_i(d))) across all ranking lists.

RRF_K = 60  # Standard RRF constant (controls how much rank matters vs presence)


def reciprocal_rank_fusion(rankings, k=RRF_K):
    """Combine multiple ranked lists using RRF.
    
    Args:
        rankings: list of lists, each inner list is [(doc_index, score), ...] 
                  ordered by score descending
        k: RRF constant (default 60)
    
    Returns:
        dict of {doc_index: rrf_score}, sorted by score descending
    """
    rrf_scores = {}
    for ranking in rankings:
        for rank, (doc_idx, _score) in enumerate(ranking):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
            rrf_scores[doc_idx] += 1.0 / (k + rank + 1)  # rank is 0-based, so +1
    return rrf_scores


def hybrid_search(query, doc_type="law", top_k=50, hyde_doc=None):
    """Hybrid FAISS semantic + BM25 keyword search with RRF fusion + reranking.
    
    Pipeline:
    1. FAISS search (semantic) → top_k candidates with scores
    2. BM25 search (keyword) → top_k candidates with scores
    3. RRF fusion → merged ranking
    4. (DISABLED) Return RRF-ordered results directly
    """
    if doc_type == "law":
        f_index = faiss_law_index
        b_index = bm25_law_index
        docs = laws_documents
        doc_types = laws_doc_types
    else:
        f_index = faiss_court_index
        b_index = bm25_court_index
        docs = courts_documents
        doc_types = courts_doc_types

    # ---- FAISS semantic search ----
    if hyde_doc:
        faiss_query = hyde_doc
        prompt = CONFIG["prompt_hyde_law"] if doc_type == "law" else CONFIG["prompt_hyde_court"]
    else:
        faiss_query = query
        prompt = CONFIG["prompt_query_law"] if doc_type == "law" else CONFIG["prompt_query_court"]

    q_vec = _st_model.encode([faiss_query], prompt=prompt, normalize_embeddings=True).astype('float32')
    faiss_scores, faiss_indices = f_index.search(q_vec, top_k)

    faiss_ranking = []
    for score, idx in zip(faiss_scores[0], faiss_indices[0]):
        if idx >= 0 and score > 0:
            faiss_ranking.append((int(idx), float(score)))

    # ---- BM25 keyword search ----
    bm25_query_tokens = _tokenize_german(query)
    bm25_scores = b_index.get_scores(bm25_query_tokens)
    # Get top_k by BM25 score
    bm25_top_indices = bm25_scores.argsort()[::-1][:top_k]
    bm25_ranking = [(int(idx), float(bm25_scores[idx])) for idx in bm25_top_indices if bm25_scores[idx] > 0]

    # ---- RRF Fusion ----
    rrf_scores = reciprocal_rank_fusion([faiss_ranking, bm25_ranking], k=RRF_K)

    # Sort by RRF score, take top candidates for reranking
    rerank_pool_size = min(top_k, len(rrf_scores))
    sorted_indices = sorted(rrf_scores.keys(), key=lambda idx: rrf_scores[idx], reverse=True)[:rerank_pool_size]

    results = []
    for idx in sorted_indices:
        doc = docs[idx].copy()
        doc["_rrf_score"] = rrf_scores[idx]
        doc["_type"] = str(doc_types[idx])
        results.append(doc)

    if _query_count < _VERBOSE_FIRST_N:
        n_faiss = len(faiss_ranking)
        n_bm25 = len(bm25_ranking)
        top_cits = [r.get("citation", "")[:30] for r in results[:3]]
        print(f"    [Hybrid {doc_type}] FAISS={n_faiss}, BM25={n_bm25} → RRF top-{len(results)}: {top_cits}")


    # ---- Per-search reranking DISABLED ----
    # WHY: Reranker was ACTIVELY HARMFUL here — promoted wrong codes (JStPO, ZISG)
    # over correct StPO articles. Agent keyword queries are not natural-language
    # questions, so cross-encoders/rerankers scramble the good RRF order.
    # Only rerank at the FINAL stage with the original query from val/test CSV.

    return results


def prf_hybrid_search(query, doc_type, top_k):
    """PRF→HyDE pipeline with hybrid FAISS+BM25+RRF retrieval.
    
    1. Initial hybrid search (FAISS+BM25+RRF) with raw query
    2. Extract top-K results as PRF context
    3. Generate HyDE doc grounded in corpus vocabulary
    4. Final hybrid search with HyDE doc + reranking
    
    Returns: (results, hyde_doc)
    """
    prf_top_k = CONFIG["prf_top_k"]

    # Step 1: Initial hybrid search (no HyDE, no reranking — just for PRF context)
    if doc_type == "law":
        f_index = faiss_law_index
        b_index = bm25_law_index
        docs = laws_documents
        prompt = CONFIG["prompt_query_law"]
    else:
        f_index = faiss_court_index
        b_index = bm25_court_index
        docs = courts_documents
        prompt = CONFIG["prompt_query_court"]

    # FAISS for PRF
    q_vec = _st_model.encode([query], prompt=prompt, normalize_embeddings=True).astype('float32')
    faiss_scores, faiss_indices = f_index.search(q_vec, prf_top_k * 2)
    faiss_ranking = [(int(idx), float(s)) for s, idx in zip(faiss_scores[0], faiss_indices[0]) if idx >= 0 and s > 0]

    # BM25 for PRF
    bm25_tokens = _tokenize_german(query)
    bm25_raw = b_index.get_scores(bm25_tokens)
    bm25_top = bm25_raw.argsort()[::-1][:prf_top_k * 2]
    bm25_ranking = [(int(idx), float(bm25_raw[idx])) for idx in bm25_top if bm25_raw[idx] > 0]

    # RRF for PRF
    prf_rrf = reciprocal_rank_fusion([faiss_ranking, bm25_ranking], k=RRF_K)
    prf_sorted = sorted(prf_rrf.keys(), key=lambda idx: prf_rrf[idx], reverse=True)[:prf_top_k]
    prf_results = [docs[idx] for idx in prf_sorted]

    if _query_count < _VERBOSE_FIRST_N:
        prf_cits = [r.get("citation", "")[:25] for r in prf_results[:3]]
        print(f"    [PRF {doc_type}] hybrid initial → {len(prf_results)} docs: {prf_cits}")

    # Step 2: Extract text snippets for HyDE context
    prf_snippets = []
    for r in prf_results[:prf_top_k]:
        cit = r.get("citation", "")
        text = r.get("text", "")[:800]
        prf_snippets.append(f"{cit}: {text}")

    # Step 3: Generate HyDE doc grounded in corpus context
    hyde_doc = generate_hypothetical_document(query, doc_type=doc_type, prf_snippets=prf_snippets)

    # Step 4: Final hybrid search with HyDE doc + full reranking
    results = hybrid_search(query, doc_type=doc_type, top_k=top_k, hyde_doc=hyde_doc)

    return results, hyde_doc


class LawSearchTool:
    """Hybrid FAISS+BM25+RRF law search with PRF→HyDE."""
    name = "search_laws"
    description = "Search Swiss federal laws (SR) by keywords. Input: German legal search terms."

    def __init__(self, top_k=50, max_excerpt_length=300):
        self.top_k = top_k
        self.max_excerpt_length = max_excerpt_length
        self._last_results = []

    def __call__(self, query):
        return self.run(query)

    def run(self, query):
        if not query or not query.strip():
            self._last_results = []
            return "Error: Empty query."
        if _query_count < _VERBOSE_FIRST_N:
            print(f"  \U0001f50d search_laws(\"{query[:60]}...\")")

        results, hyde_doc = prf_hybrid_search(query, doc_type="law", top_k=self.top_k)
        self._last_hyde_doc = hyde_doc
        self._last_results = results

        if not results:
            return f"No relevant federal laws found for: \'{query}\'"
        formatted = []
        for doc in results:
            cit = doc.get("citation", "Unknown")
            text = doc.get("text", "")[:self.max_excerpt_length]
            formatted.append(f"- [{doc.get('_type','')}] {cit}: {text}")
        return "\n".join(formatted)

    def get_last_citations(self):
        return [d.get("citation", "") for d in self._last_results if d.get("citation")]


class CourtSearchTool:
    """Hybrid FAISS+BM25+RRF court search with PRF→HyDE."""
    name = "search_courts"
    description = "Search Swiss Federal Court decisions (BGE) by keywords. Input: German legal search terms."

    def __init__(self, top_k=50, max_excerpt_length=300):
        self.top_k = top_k
        self.max_excerpt_length = max_excerpt_length
        self._last_results = []

    def __call__(self, query):
        return self.run(query)

    def run(self, query):
        if not query or not query.strip():
            self._last_results = []
            return "Error: Empty query."
        if _query_count < _VERBOSE_FIRST_N:
            print(f"  \U0001f50d search_courts(\"{query[:60]}...\")")

        results, hyde_doc = prf_hybrid_search(query, doc_type="court", top_k=self.top_k)
        self._last_hyde_doc = hyde_doc
        self._last_results = results

        if not results:
            return f"No relevant court decisions found for: \'{query}\'"
        formatted = []
        for doc in results:
            cit = doc.get("citation", "Unknown")
            text = doc.get("text", "")[:self.max_excerpt_length]
            formatted.append(f"- [{doc.get('_type','')}] {cit}: {text}")
        return "\n".join(formatted)

    def get_last_citations(self):
        return [d.get("citation", "") for d in self._last_results if d.get("citation")]


# ---- Register tools ----
law_tool = LawSearchTool(top_k=CONFIG["top_k_laws"])
court_tool = CourtSearchTool(top_k=CONFIG["top_k_courts"])
TOOLS = {"search_laws": law_tool, "search_courts": court_tool}

print(f"Hybrid search tools ready (FAISS + BM25 + RRF → reranking)")
print(f"  RRF k={RRF_K}")
print(f"  top_k: laws={CONFIG['top_k_laws']}, courts={CONFIG['top_k_courts']}")
print(f"  PRF: {CONFIG['prf_top_k']} initial hybrid results → HyDE → final hybrid search")


In [ ]:
# === CELL 10: STRUCTURED REACT AGENT (GBNF grammar + Pydantic) ===
# Run 6 fixes:
# - Prompt stripped to 2 examples (from 4) — less repetition
# - Query truncated to 40 chars + article numbers stripped
# - Guardrails: skip duplicate queries, force tool alternation

from pydantic import BaseModel, field_validator
from typing import Literal
from llama_cpp import LlamaGrammar
import json as _json

# ---- Pydantic model for agent output ----
class AgentAction(BaseModel):
    thought: str
    action: Literal["search_laws", "search_courts", "done"]
    query: str = ""

    @field_validator("query")
    @classmethod
    def clean_query(cls, v):
        # Strip any leaked syntax that somehow got through
        v = re.sub(r"Action\s*Input\s*:.*", "", v, flags=re.I).strip()
        v = re.sub(r"Tool\s+search_\w+:.*", "", v, flags=re.I).strip()
        v = re.sub(r"\[INST\].*", "", v).strip()
        # Strip article numbers — they confuse the embedding model
        v = re.sub(r"Art\.?\s*\d+\s*(Abs\.?\s*\d+)?\s*(lit\.?\s*[a-z])?\s*", "", v).strip()
        # Soft truncate — instruction-guided embeddings handle longer queries well
        return v[:500]


# ---- GBNF Grammar: forces valid JSON with exactly one action per turn ----
_GBNF_LINES = [
    r'root ::= "{" ws "\"thought\"" ws ":" ws string "," ws "\"action\"" ws ":" ws action-val "," ws "\"query\"" ws ":" ws string "}" ws',
    r'action-val ::= "\"search_laws\"" | "\"search_courts\"" | "\"done\""',
    r'string ::= "\"" chars "\""',
    r'chars ::= char*',
    r'char ::= [^"\\] | "\\" escape-char',
    r'escape-char ::= "\"" | "\\" | "/" | "n" | "t" | "r"',
    r'ws ::= [ \t\n]*',
]
AGENT_GRAMMAR_STR = "\n".join(_GBNF_LINES)
AGENT_GRAMMAR = LlamaGrammar.from_string(AGENT_GRAMMAR_STR)


# ---- System prompt (German, 2 examples, short queries) ----

# ---- Load Swiss Legal Routing Guide from external .txt files ----
# Keeps this cell clean; edit routing_guide_laws.txt and routing_guide_courts.txt directly.
# On Kaggle: upload the .txt files as a utility dataset named "routing-guides"
_GUIDE_SEARCH_PATHS = [
    Path("."),                                          # Kaggle working dir / local CWD
    Path("/kaggle/input/routing-guides"),               # Kaggle utility dataset
    DATA_PATH,                                         # Competition data dir (fallback)
    Path("../data"),                                   # Local dev (notebook in notebooks/)
]

def _find_guide(filename):
    for p in _GUIDE_SEARCH_PATHS:
        candidate = p / filename
        if candidate.exists():
            return candidate
    return None

_laws_guide_path = _find_guide("routing_guide_laws.txt")
_courts_guide_path = _find_guide("routing_guide_courts.txt")

if _laws_guide_path:
    ROUTING_GUIDE_LAWS = _laws_guide_path.read_text(encoding="utf-8")
    print(f"  Loaded {_laws_guide_path} ({len(ROUTING_GUIDE_LAWS)} chars)")
else:
    ROUTING_GUIDE_LAWS = ""
    print("  WARNING: routing_guide_laws.txt not found — guide disabled")

if _courts_guide_path:
    ROUTING_GUIDE_COURTS = _courts_guide_path.read_text(encoding="utf-8")
    print(f"  Loaded {_courts_guide_path} ({len(ROUTING_GUIDE_COURTS)} chars)")
else:
    ROUTING_GUIDE_COURTS = ""
    print("  WARNING: routing_guide_courts.txt not found — guide disabled")

AGENT_SYSTEM_PROMPT = f"""You are a Swiss legal citation retrieval agent. Your job: generate search queries to find relevant Swiss legal citations.

You have 2 tools:
- search_laws: searches Swiss federal statutes (SR collection)
- search_courts: searches Swiss Federal Court decisions (BGE + unpublished cases)

You MUST respond with a single JSON object on each turn:
{{"thought": "your reasoning", "action": "search_laws|search_courts|done", "query": "German legal search terms"}}

When action is "done", set query to "" — this signals you are finished searching.

Strategy:
- Search BOTH laws and courts (alternate between them)
- Write queries in GERMAN using Swiss legal terminology (the corpus is German)
- Each query should target a different aspect of the legal question
- After 3-4 searches covering both sources, use action "done"
- Use the routing guide below to pick the right tool and keywords

Available law types: {LAW_TYPES_FOR_PROMPT}
Court types: {COURT_TYPES_FOR_PROMPT}

{ROUTING_GUIDE_LAWS}

{ROUTING_GUIDE_COURTS}

Example turns for query "What are requirements for a valid contract?":
Turn 1: {{"thought": "Suche Vertragsentstehung im Obligationenrecht", "action": "search_laws", "query": "Vertragsentstehung gegenseitige Zustimmung Verpflichtung Obligationenrecht"}}
Turn 2: {{"thought": "Bundesgerichtsentscheide zur Vertragsgueltigkeit", "action": "search_courts", "query": "Vertrag gueltig Voraussetzungen Zustimmung Willenserklärung"}}
Turn 3: {{"thought": "Willensmängel bei Vertragsschluss", "action": "search_laws", "query": "Willensmängel Irrtum Täuschung Furchterregung Vertrag"}}
Turn 4: {{"thought": "Genug gesucht in beiden Quellen", "action": "done", "query": ""}}"""


# ---- Observation formatting (summary only — no raw text) ----




def format_observation(tool_name, results, top_n=5):
    """Format tool results as a compact summary for the agent context."""
    count = len(results)
    if count == 0:
        return f"[{tool_name}: 0 results]"
    top_citations = [r.get("citation", "?")[:40] for r in results[:top_n]]
    cit_str = ", ".join(f'"{c}"' for c in top_citations)
    return f"[{tool_name}: {count} results. Top {min(top_n, count)}: {cit_str}]"


# ---- Extract explicit citations from query text (Quick win #4) ----
def extract_citations_from_query(query, laws_docs, courts_docs):
    """Regex-extract explicit citations mentioned in the query. FREE precision."""
    law_cit_set = {d["citation"] for d in laws_docs}
    court_cit_set = {d["citation"] for d in courts_docs}
    found = []
    # Pattern: "Art. 123 [Abs. 4] CODE" (Swiss law citations)
    for m in re.finditer(r'Art\.?\s*(\d+)\s*(?:Abs\.?\s*(\d+)\s*)?([A-Z]{2,}[a-z]?)', query):
        article_num, abs_num, code = m.group(1), m.group(2), m.group(3)
        # Try to find matching citation in corpus
        for cit in law_cit_set:
            if f"Art. {article_num}" in cit and code in cit:
                if abs_num and f"Abs. {abs_num}" in cit:
                    found.append(cit)
                    break
                elif not abs_num:
                    found.append(cit)
                    break
    # Pattern: "BGE 123 IV 567"
    for m in re.finditer(r'BGE\s+\d+\s+[IVX]+\s+\d+', query):
        candidate = m.group(0)
        for cit in court_cit_set:
            if candidate in cit:
                found.append(cit)
                break
    return list(dict.fromkeys(found))  # dedupe preserving order


# ---- Main agent loop ----
def run_agent(query, verbose=False):
    """Run structured ReAct agent. Returns (citations_list, logs_list)."""
    all_results = []   # Full result dicts (with scores) for final reranking
    all_citations = []  # Legacy: just citation strings
    logs = []
    history = []  # list of (action_str, observation_summary) tuples
    first_hyde_doc = None  # Quick win #2: capture first HyDE for final reranking
    _seen_queries = set()  # Guardrail: skip duplicate queries
    _last_tools = []  # Guardrail: track tool sequence for forced alternation

    for iteration in range(CONFIG["max_iterations"]):
        # Build prompt
        prompt = f"[INST] {AGENT_SYSTEM_PROMPT}\n\nQuery: {query}\n\n"
        if history:
            prompt += "Previous searches:\n"
            for h_action, h_obs in history:
                prompt += f"- {h_action} -> {h_obs}\n"
            prompt += "\n"
        prompt += "Respond with your next action as JSON: [/INST]\n"

        # Truncate if needed (keep system + query + last 3 history items)
        if len(prompt) > CONFIG["max_conversation_chars"]:
            prompt = f"[INST] {AGENT_SYSTEM_PROMPT}\n\nQuery: {query}\n\n"
            if history:
                prompt += "Previous searches (recent):\n"
                for h_action, h_obs in history[-3:]:
                    prompt += f"- {h_action} -> {h_obs}\n"
                prompt += "\n"
            prompt += "Respond with your next action as JSON: [/INST]\n"

        # Generate with grammar constraint
        try:
            response = llm(
                prompt,
                max_tokens=CONFIG["max_tokens"],
                temperature=CONFIG["temperature"],
                grammar=AGENT_GRAMMAR,
                stop=["[INST]", "</s>"],
            )["choices"][0]["text"].strip()
        except Exception as e:
            if verbose:
                print(f"    [ERROR] LLM generation failed: {e}")
            break

        # Parse JSON (grammar guarantees valid JSON structure)
        try:
            parsed = _json.loads(response)
            action = AgentAction(**parsed)
        except (ValueError, Exception) as e:
            if verbose:
                print(f"    [ERROR] Parse failed: {e} | raw: {response[:100]}")
            break

        if verbose:
            print(f"  [Iter {iteration+1}] thought=\"{action.thought[:80]}\" "
                  f"action={action.action} query=\"{action.query[:40]}\"")

        # Handle "done" action — force minimum 4 searches (2 law + 2 court)
        if action.action == "done":
            searches_done = len([t for t in _last_tools if t in ("search_laws", "search_courts")])
            law_searches = _last_tools.count("search_laws")
            court_searches = _last_tools.count("search_courts")
            if searches_done < 4:
                # Force more searches — alternate to cover both sources
                if law_searches < 2:
                    action.action = "search_laws"
                elif court_searches < 2:
                    action.action = "search_courts"
                else:
                    action.action = "search_laws" if law_searches <= court_searches else "search_courts"
                if verbose:
                    print(f"    [GUARDRAIL] Too few searches ({searches_done}/4 min) — forcing {action.action}")
                # Generate a broader query for the forced search
                action.query = query[:200]  # Use original query as fallback
            else:
                if verbose:
                    print(f"    -> Agent signaled done after {iteration+1} iterations ({searches_done} searches)")
                break

        # ---- GUARDRAIL: Force alternation (can't call same tool 3x in a row) ----
        if len(_last_tools) >= 2 and all(t == action.action for t in _last_tools[-2:]):
            forced_tool = "search_courts" if action.action == "search_laws" else "search_laws"
            if verbose:
                print(f"    [GUARDRAIL] Forced alternation: {action.action} -> {forced_tool}")
            action.action = forced_tool

        # ---- GUARDRAIL: Skip duplicate queries ----
        query_key = f"{action.action}:{action.query}"
        if query_key in _seen_queries:
            if verbose:
                print(f"    [GUARDRAIL] Skipping duplicate: {action.query[:40]}")
            history.append((f'{action.action}("{action.query[:30]}")', '[SKIPPED: duplicate]'))
            _last_tools.append(action.action)
            continue
        _seen_queries.add(query_key)

        # Execute tool
        tool = TOOLS.get(action.action)
        if not tool:
            if verbose:
                print(f"    [WARN] Unknown tool: {action.action}")
            break

        observation = tool(action.query)
        obs_citations = tool.get_last_citations()
        all_citations.extend(obs_citations)
        all_results.extend(tool._last_results)  # Keep full results for final rerank

        # Quick win #2: Capture first HyDE doc for final reranking (German, free)
        if first_hyde_doc is None:
            first_hyde_doc = getattr(tool, '_last_hyde_doc', None)

        _last_tools.append(action.action)

        # Format compact observation for history
        obs_summary = format_observation(action.action, tool._last_results, top_n=5)
        history.append((f'{action.action}("{action.query[:50]}")', obs_summary))

        if verbose:
            print(f"    [{action.action}] -> {len(obs_citations)} citations")

        logs.append({
            "iteration": iteration,
            "thought": action.thought,
            "action": action.action,
            "query": action.query,
            "n_citations": len(obs_citations),
        })

    # Deduplicate citations (preserve order, keep best result per citation)
    seen = {}
    for r in all_results:
        cit = r.get("citation", "")
        if not cit:
            continue
        # Keep the result with highest rerank score (or FAISS score as fallback)
        score = r.get("_rerank_score", r.get("_score", 0))
        if cit not in seen or score > seen[cit].get("_rerank_score", seen[cit].get("_score", 0)):
            seen[cit] = r
    deduped_results = list(seen.values())

    # ---- Final reranking with Qwen3-Reranker (generative yes/no) ----
    # WHY only here (not per-search): The original query from val/test CSV is natural
    # language (English). Rerankers work on (question, document) pairs.
    # Agent keyword queries like "Untersuchungshaft StPO" are NOT questions.
    final_top_n = CONFIG.get("final_rerank_top_n", 50)
    if _reranker and CONFIG.get("rerank_enabled") and len(deduped_results) > 0:
        rerank_query = query  # original natural-language query from CSV
        pairs = [(rerank_query, r.get("text", "")[:TEXT_TRUNCATE]) for r in deduped_results]
        final_scores = _reranker.predict(pairs, show_progress_bar=False)
        for doc, fs in zip(deduped_results, final_scores):
            doc["_final_rerank_score"] = float(fs)
        deduped_results.sort(key=lambda d: d["_final_rerank_score"], reverse=True)
        
        # Score cutoff — Qwen3 outputs P(yes) in [0,1]. 0.1 = "10% confident relevant"
        # WHY 0.1: Gold has 42 citations, need high recall. Only filter obvious garbage.
        SCORE_CUTOFF = 0.1
        before_cutoff = len(deduped_results)
        deduped_results = [r for r in deduped_results if r.get("_final_rerank_score", 0) > SCORE_CUTOFF]
        deduped_results = deduped_results[:final_top_n]
        if verbose:
            print(f"  [Final rerank] {len(seen)} unique -> cutoff dropped {before_cutoff - len(deduped_results)} -> kept {len(deduped_results)}")
    elif verbose and len(deduped_results) > 0:
        print(f"  [Final] {len(deduped_results)} unique citations (no final rerank needed)")

    deduped = [r.get("citation", "") for r in deduped_results]

    # Prepend any explicit citations extracted via regex
    explicit_cits = extract_citations_from_query(query, laws_documents, courts_documents)
    if explicit_cits:
        if verbose:
            print(f"  [Regex] Extracted {len(explicit_cits)} explicit citations from query: {explicit_cits}")
        deduped = explicit_cits + [c for c in deduped if c not in explicit_cits]

    if verbose:
        print(f"  [Final] {len(deduped)} citations after expansion")

    return deduped, logs


# Quick sanity check
print(f"Agent system prompt: {len(AGENT_SYSTEM_PROMPT):,} chars")
print(f"Agent grammar: GBNF constrained (search_laws|search_courts|done)")
print(f"Agent ready — max {CONFIG['max_iterations']} iterations, 1 tool/iter")
print(f"Guardrails: min 4 searches, no article nums, dedup, forced alternation, soft truncate 500")


In [ ]:
# === CELL 13: LOCAL EVALUATION (with checkpointing) ===
from collections.abc import Sequence as SeqType

CHECKPOINT_EVERY = 5  # save progress every N val queries

def citation_f1(predicted, gold):
    pred_set, gold_set = set(predicted), set(gold)
    if not pred_set and not gold_set:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
    if not pred_set:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    if not gold_set:
        return {"precision": 0.0, "recall": 1.0, "f1": 0.0}
    tp = len(pred_set & gold_set)
    p = tp / len(pred_set)
    r = tp / len(gold_set)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": p, "recall": r, "f1": f1}


VAL_CSV = DATA_PATH / "val.csv"
VAL_CHECKPOINT_PATH = OUTPUT_PATH / "val_predictions_checkpoint.pkl"

if VAL_CSV.exists():
    val_df = pd.read_csv(VAL_CSV)
    if "gold_citations" in val_df.columns:
        print(f"Running validation on {len(val_df)} queries...")

        # FRESH RUN — pipeline logic changed (quick wins applied)
        val_predictions = []

        for i, (_, row) in enumerate(tqdm(val_df.iterrows(), total=len(val_df), desc="Val agent")):
            cits, _ = run_agent(row["query"], verbose=(i < 3))
            val_predictions.append(cits)
            if (i + 1) % CHECKPOINT_EVERY == 0:
                with open(VAL_CHECKPOINT_PATH, 'wb') as f:
                    pickle.dump(val_predictions, f)
                _save_hyde_cache()
                _save_checkpoint(VAL_CHECKPOINT_PATH, HYDE_CACHE_PATH)
                print(f"  [Val checkpoint] {len(val_predictions)}/{len(val_df)}")

        # Final save
        with open(VAL_CHECKPOINT_PATH, 'wb') as f:
            pickle.dump(val_predictions, f)
        _save_hyde_cache()
        _save_checkpoint(VAL_CHECKPOINT_PATH, HYDE_CACHE_PATH)

        val_golds = [str(row["gold_citations"]).split(";") for _, row in val_df.iterrows()]
        val_golds = [[c.strip() for c in g if c.strip()] for g in val_golds]

        f1_scores = [citation_f1(p, g)["f1"] for p, g in zip(val_predictions, val_golds)]
        macro_f1_val = sum(f1_scores) / len(f1_scores)
        print(f"\nMacro F1: {macro_f1_val:.4f}")
        print(f"Per-query F1: {[f'{s:.3f}' for s in f1_scores]}")
    else:
        print("val.csv has no gold_citations column — skipping evaluation.")
else:
    print("No val.csv found — skipping evaluation.")

In [ ]:
# === CELL 11: RUN PREDICTIONS ON TEST SET (with checkpointing + progress) ===
from tqdm import tqdm

test_df = pd.read_csv(TEST_CSV)
print(f"Loaded {len(test_df)} test queries")

CHECKPOINT_PATH = OUTPUT_PATH / "predictions_checkpoint.pkl"
CHECKPOINT_EVERY = 25  # save progress every N queries

# FRESH RUN — pipeline logic changed (quick wins applied), don't resume old predictions
predictions = []
done_ids = set()
print(f"  🆕 Fresh run (quick wins applied — no resume from old checkpoint)")

remaining = test_df[~test_df["query_id"].isin(done_ids)]
print(f"  Remaining: {len(remaining)} queries")
print(f"  Config: max_iterations={CONFIG['max_iterations']}, top_k={CONFIG['top_k_laws']}, hyde={'ON' if CONFIG['hyde_enabled'] else 'OFF'}")
print(f"  Checkpoint every {CHECKPOINT_EVERY} queries → {CHECKPOINT_PATH.name}")
print()

# Running stats
_total_citations = sum(len(p["predicted_citations"].split(";")) for p in predictions if p["predicted_citations"])
_t_start = time.time()

for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc="Running agent")):
    query_id = row["query_id"]
    query_text = row["query"]

    # Detailed output for first N
    if _query_count < _VERBOSE_FIRST_N:
        print(f"\n{'='*70}")
        print(f"  Query #{_query_count+1}: \"{query_text[:100]}...\"")
        print(f"{'='*70}")

    raw_citations, _ = run_agent(query_text, verbose=(_query_count < _VERBOSE_FIRST_N))
    _query_count += 1
    _total_citations += len(raw_citations)

    predictions.append({
        "query_id": query_id,
        "predicted_citations": ";".join(raw_citations),
    })

    # Per-query summary (always shown for first N, then every 25)
    if _query_count <= _VERBOSE_FIRST_N:
        print(f"  ✅ Found {len(raw_citations)} citations: {raw_citations[:5]}")
        if _query_count == _VERBOSE_FIRST_N:
            print(f"\n  --- Switching to quiet mode (summaries every {CHECKPOINT_EVERY} queries) ---\n")

    # Checkpoint every N queries
    if (i + 1) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump(predictions, f)
        _save_hyde_cache()
        _save_checkpoint(CHECKPOINT_PATH, HYDE_CACHE_PATH)
        elapsed = time.time() - _t_start
        avg_cits = _total_citations / len(predictions) if predictions else 0
        qps = (i + 1) / elapsed if elapsed > 0 else 0
        eta_min = (len(remaining) - i - 1) / qps / 60 if qps > 0 else 0
        print(f"  💾 Checkpoint {len(predictions)}/{len(test_df)} | "
              f"avg {avg_cits:.1f} cits/query | {qps:.2f} q/s | ETA ~{eta_min:.0f} min | "
              f"HyDE cache: {len(_hyde_cache)}")

# Final save
with open(CHECKPOINT_PATH, 'wb') as f:
    pickle.dump(predictions, f)
_save_checkpoint(CHECKPOINT_PATH, HYDE_CACHE_PATH)

predictions_df = pd.DataFrame(predictions)
elapsed_total = time.time() - _t_start
print(f"\n{'='*70}")
print(f"  ✅ DONE: {len(predictions_df)} predictions in {elapsed_total/60:.1f} min")
print(f"  Avg citations/query: {predictions_df['predicted_citations'].str.count(';').mean() + 1:.1f}")
print(f"  HyDE cache: {len(_hyde_cache)} entries saved to disk")
print(f"  Queries with 0 citations: {(predictions_df['predicted_citations'] == '').sum()}")

print(f"{'='*70}")

In [ ]:
# === CELL 12: SAVE SUBMISSION ===
submission_path = OUTPUT_PATH / "submission.csv"
predictions_df.to_csv(submission_path, index=False)
_save_checkpoint(submission_path)
print(f"Submission saved to: {submission_path}")
print(f"Total predictions: {len(predictions_df)}")
predictions_df.head()

In [ ]:
# === CELL 14: FINAL SAVE ===
print("Final push to persistent dataset...")
_final_save()

print(f"\n{'='*70}")
print(f"  📥 ALL GENERATED FILES:")
print(f"     • submission.csv — upload directly to competition")
print(f"     • *.pkl files — all caches (embeddings, checkpoints, hyde)")
print(f"     Location: /kaggle/working/ (current session)")
print(f"     PERMANENT: kaggle.com/datasets/{DATASET_SLUG}")
print(f"  💡 Checkpoints pushed to dataset after every 25 queries.")
print(f"     Even if notebook crashed earlier, your data is safe.")
print(f"{'='*70}")